# 📊 Phase 2: Exploratory Data Analysis (EDA)

**Indian Housing Price Prediction**

This notebook performs a comprehensive EDA on the raw housing dataset to understand:
- Data distribution and quality
- Correlations between features and the target (`Price_in_Lakhs`)
- Geospatial price patterns across cities and states
- Impact of categorical features and amenities on price

---

## 1️⃣ Setup & Data Loading

We start by importing the required libraries, configuring plot styles, and loading the dataset using the existing `data_ingestion` module from `src/`.

In [ ]:
# Core libraries
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plot aesthetics
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.0)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Resolve project root so we can import from src/
project_root = Path.cwd()
for candidate in [project_root, project_root.parent, project_root / 'indian_housing']:
    if (candidate / 'src').exists() and (candidate / 'data' / 'raw' / 'india_housing_prices.csv').exists():
        project_root = candidate
        break

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Load data using the existing ingestion module
from src.data_ingestion import load_data

csv_path = project_root / 'data' / 'raw' / 'india_housing_prices.csv'
df = load_data(csv_path)

# Quick confirmation
print(f'Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

## 2️⃣ Data Overview

A high-level look at the dataset structure — column names, data types, and a preview of the first few rows.

In [ ]:
# Column names and data types
print('Column Overview:')
print('-' * 60)
for col in df.columns:
    print(f'  {col:35s}  {str(df[col].dtype):10s}  '
          f'unique={df[col].nunique():>6d}')

In [ ]:
# Preview the first 5 rows
df.head()

In [ ]:
# Preview the last 5 rows
df.tail()

In [ ]:
# Summary of numeric columns
df.describe().T

In [ ]:
# Summary of categorical columns
df.describe(include='object').T

## 3️⃣ Target Variable Analysis — `Price_in_Lakhs`

The target variable is the property price in Lakhs (₹1 Lakh = ₹100,000). We examine its distribution, central tendency, spread, skewness, and potential outliers.

In [ ]:
# Descriptive statistics for the target
price_stats = df['Price_in_Lakhs'].describe()
print('Price_in_Lakhs — Descriptive Statistics:')
print(price_stats.round(2))

# Skewness
skewness = df['Price_in_Lakhs'].skew()
print(f'\nSkewness: {skewness:.4f}  (close to 0 = symmetric)')

In [ ]:
# Distribution histogram with KDE overlay
fig, ax = plt.subplots(figsize=(12, 5))

ax1 = ax
sns.histplot(df['Price_in_Lakhs'], bins=50, kde=True, color='steelblue', ax=ax1)
ax1.set_title('Distribution of Price_in_Lakhs', fontsize=14, fontweight='bold')
ax1.set_xlabel('Price (Lakhs)')
ax1.set_ylabel('Frequency')

# Add mean and median lines
mean_val = df['Price_in_Lakhs'].mean()
median_val = df['Price_in_Lakhs'].median()
ax1.axvline(mean_val, color='red', linestyle='--', linewidth=1.5, label=f'Mean = ₹{mean_val:.1f}L')
ax1.axvline(median_val, color='green', linestyle='--', linewidth=1.5, label=f'Median = ₹{median_val:.1f}L')
ax1.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Outlier detection using the IQR method
Q1 = df['Price_in_Lakhs'].quantile(0.25)
Q3 = df['Price_in_Lakhs'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['Price_in_Lakhs'] < lower_bound) | (df['Price_in_Lakhs'] > upper_bound)]

print(f'IQR Method — Lower bound: ₹{lower_bound:.2f}L, Upper bound: ₹{upper_bound:.2f}L')
print(f'Outliers detected: {len(outliers):,} rows ({len(outliers)/len(df)*100:.2f}%)')
print(f'  - Below lower bound: {(df["Price_in_Lakhs"] < lower_bound).sum():,}')
print(f'  - Above upper bound: {(df["Price_in_Lakhs"] > upper_bound).sum():,}')

In [ ]:
# Box plot to visualise outliers
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(x=df['Price_in_Lakhs'], ax=ax, color='steelblue')
ax.set_title('Box Plot of Price_in_Lakhs (Outlier Detection)', fontsize=14, fontweight='bold')
ax.set_xlabel('Price (Lakhs)')
plt.tight_layout()
plt.show()

In [ ]:
# Compare original vs. log-transformed distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original distribution
sns.histplot(df['Price_in_Lakhs'], bins=50, kde=True, color='steelblue', ax=axes[0])
axes[0].set_title(f'Original (Skewness = {df["Price_in_Lakhs"].skew():.3f})')
axes[0].set_xlabel('Price (Lakhs)')

# Log-transformed distribution
log_price = np.log1p(df['Price_in_Lakhs'])
sns.histplot(log_price, bins=50, kde=True, color='coral', ax=axes[1])
axes[1].set_title(f'Log1p Transformed (Skewness = {log_price.skew():.3f})')
axes[1].set_xlabel('log(1 + Price)')

plt.suptitle('Original vs. Log-Transformed Price Distribution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Original skewness:  {df["Price_in_Lakhs"].skew():.4f}')
print(f'Log1p skewness:     {log_price.skew():.4f}')

## 4️⃣ Numeric Correlation Analysis

We examine how numeric features correlate with each other and with the target variable. This helps identify which features carry the most predictive signal.

In [ ]:
# Correlation matrix for all numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df[numeric_cols].corr()

# Heatmap
fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Heatmap — Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation of each numeric feature with the target
target_corr = corr_matrix['Price_in_Lakhs'].sort_values(ascending=False)
print('Correlation with Price_in_Lakhs (sorted):')
print(target_corr.round(4))

In [ ]:
# Scatter plot: Size_in_SqFt vs. Price
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=df, x='Size_in_SqFt', y='Price_in_Lakhs', alpha=0.3, s=10, ax=ax)
sns.regplot(data=df, x='Size_in_SqFt', y='Price_in_Lakhs', scatter=False, color='red', ax=ax)
ax.set_title('Size_in_SqFt vs. Price_in_Lakhs', fontsize=14, fontweight='bold')
ax.set_xlabel('Size (SqFt)')
ax.set_ylabel('Price (Lakhs)')
plt.tight_layout()
plt.show()

print(f'Correlation: {df["Size_in_SqFt"].corr(df["Price_in_Lakhs"]):.4f}')

In [ ]:
# Scatter plot: BHK vs. Price
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df, x='BHK', y='Price_in_Lakhs', ax=ax, palette='Set2')
ax.set_title('Price Distribution by BHK', fontsize=14, fontweight='bold')
ax.set_xlabel('BHK (Bedrooms, Hall, Kitchen)')
ax.set_ylabel('Price (Lakhs)')
plt.tight_layout()
plt.show()

print('Average price by BHK:')
print(df.groupby('BHK')['Price_in_Lakhs'].mean().round(2))

In [ ]:
# Scatter plot: Price_per_SqFt vs. Price
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=df, x='Price_per_SqFt', y='Price_in_Lakhs', alpha=0.3, s=10, ax=ax)
sns.regplot(data=df, x='Price_per_SqFt', y='Price_in_Lakhs', scatter=False, color='red', ax=ax)
ax.set_title('Price_per_SqFt vs. Price_in_Lakhs', fontsize=14, fontweight='bold')
ax.set_xlabel('Price per SqFt')
ax.set_ylabel('Price (Lakhs)')
plt.tight_layout()
plt.show()

print(f'Correlation: {df["Price_per_SqFt"].corr(df["Price_in_Lakhs"]):.4f}')

## 5️⃣ Geospatial Analysis

We compare average prices across different **Cities** and **States** to understand regional pricing patterns.

In [ ]:
# Average price by City (top 15 most expensive)
city_prices = df.groupby('City')['Price_in_Lakhs'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
sns.barplot(x=city_prices.values, y=city_prices.index, ax=ax, palette='viridis')
ax.set_title('Average Price by City (All Cities)', fontsize=14, fontweight='bold')
ax.set_xlabel('Average Price (Lakhs)')
ax.set_ylabel('City')
plt.tight_layout()
plt.show()

print('Top 5 most expensive cities:')
print(city_prices.head().round(2))
print('\nBottom 5 least expensive cities:')
print(city_prices.tail().round(2))

In [ ]:
# Average price by State
state_prices = df.groupby('State')['Price_in_Lakhs'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
sns.barplot(x=state_prices.values, y=state_prices.index, ax=ax, palette='magma')
ax.set_title('Average Price by State', fontsize=14, fontweight='bold')
ax.set_xlabel('Average Price (Lakhs)')
ax.set_ylabel('State')
plt.tight_layout()
plt.show()

print('Top 5 most expensive states:')
print(state_prices.head().round(2))

In [ ]:
# Price distribution by City (box plot for top 15 cities by listing count)
top_cities = df['City'].value_counts().head(15).index.tolist()
df_top_cities = df[df['City'].isin(top_cities)]

fig, ax = plt.subplots(figsize=(16, 8))
sns.boxplot(data=df_top_cities, x='City', y='Price_in_Lakhs', ax=ax, palette='Set3')
ax.set_title('Price Distribution by City (Top 15 Cities by Listing Count)', fontsize=14, fontweight='bold')
ax.set_xlabel('City')
ax.set_ylabel('Price (Lakhs)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 6️⃣ Categorical Feature Analysis

We analyze how each categorical feature influences the target price. For each feature, we compute the average price per category and visualize it.

In [ ]:
# Helper function to plot average price by a categorical column
def plot_price_by_category(col, df, n_top=None, palette='Set2'):
    """Plot average price grouped by a categorical column."""
    grouped = df.groupby(col)['Price_in_Lakhs'].agg(['mean', 'count']).round(2)
    if n_top:
        grouped = grouped.sort_values('mean', ascending=False).head(n_top)
    else:
        grouped = grouped.sort_values('mean', ascending=False)

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(x=grouped.index, y='mean', data=grouped, ax=ax, palette=palette)
    ax.set_title(f'Average Price by {col}', fontsize=14, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Average Price (Lakhs)')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

    print(f'Average price by {col}:')
    print(grouped)
    print()

In [ ]:
# Property Type
plot_price_by_category('Property_Type', df)

In [ ]:
# Furnished Status
plot_price_by_category('Furnished_Status', df)

In [ ]:
# BHK
plot_price_by_category('BHK', df)

In [ ]:
# Facing
plot_price_by_category('Facing', df)

In [ ]:
# Owner Type
plot_price_by_category('Owner_Type', df)

In [ ]:
# Availability Status
plot_price_by_category('Availability_Status', df)

In [ ]:
# Parking Space
plot_price_by_category('Parking_Space', df)

In [ ]:
# Security
plot_price_by_category('Security', df)

In [ ]:
# Public Transport Accessibility
plot_price_by_category('Public_Transport_Accessibility', df)

In [ ]:
# Age of Property — binned for better visualization
df['Age_Bin'] = pd.cut(df['Age_of_Property'], bins=[0, 5, 10, 15, 20, 30, 100],
                        labels=['0-5', '6-10', '11-15', '16-20', '21-30', '30+'])
plot_price_by_category('Age_Bin', df)

## 7️⃣ Amenities Analysis

The `Amenities` column contains a comma-separated string of amenities. We parse it into individual binary flags and analyze the price impact of each amenity.

In [ ]:
# Parse the Amenities column into a set of unique amenities
all_amenities = set()
for amenity_str in df['Amenities'].dropna():
    for item in amenity_str.split(','):
        all_amenities.add(item.strip())

unique_amenities = sorted(all_amenities)
print(f'Unique amenities found: {len(unique_amenities)}')
print(unique_amenities)

In [ ]:
# Multi-hot encode the Amenities column
for amenity in unique_amenities:
    df[f'Has_{amenity}'] = df['Amenities'].apply(
        lambda x: 1 if amenity in str(x).split(',') else 0
    )

amenity_cols = [f'Has_{a}' for a in unique_amenities]
print(f'Created {len(amenity_cols)} binary amenity columns:')
print(amenity_cols)

# Show the encoded columns
df[amenity_cols].head()

In [ ]:
# Average price for properties WITH vs. WITHOUT each amenity
amenity_impact = []
for col in amenity_cols:
    amenity_name = col.replace('Has_', '')
    with_amenity = df[df[col] == 1]['Price_in_Lakhs']
    without_amenity = df[df[col] == 0]['Price_in_Lakhs']
    amenity_impact.append({
        'Amenity': amenity_name,
        'Avg_Price_With': round(with_amenity.mean(), 2),
        'Avg_Price_Without': round(without_amenity.mean(), 2),
        'Price_Difference': round(with_amenity.mean() - without_amenity.mean(), 2),
        'Count_With': with_amenity.count(),
    })

amenity_df = pd.DataFrame(amenity_impact).sort_values('Price_Difference', ascending=False)
print('Amenity Price Impact (sorted by price difference):')
print(amenity_df.to_string(index=False))

In [ ]:
# Visualise amenity price impact
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=amenity_df, x='Amenity', y='Price_Difference', ax=ax, palette='coolwarm')
ax.set_title('Price Impact of Each Amenity (With - Without)', fontsize=14, fontweight='bold')
ax.set_xlabel('Amenity')
ax.set_ylabel('Price Difference (Lakhs)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 8️⃣ Data Quality Check

We verify data integrity by checking for missing values, duplicate records, and any data type inconsistencies.

In [ ]:
# Missing values report
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'Missing_Count': missing, 'Missing_Pct': missing_pct})
missing_report = missing_report[missing_report['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

if missing_report.empty:
    print('✅ No missing values found in any column!')
else:
    print('⚠️  Missing values detected:')
    print(missing_report)

In [ ]:
# Duplicate records check
duplicates = df.duplicated().sum()
print(f'Duplicate rows: {duplicates:,}')
if duplicates > 0:
    print(f'  ({duplicates/len(df)*100:.2f}% of total rows)')
else:
    print('✅ No duplicate rows found!')

In [ ]:
# Data type consistency check
print('Data Types:')
print(df.dtypes)

# Check for any unexpected string values in numeric columns
numeric_cols = ['BHK', 'Size_in_SqFt', 'Price_in_Lakhs', 'Price_per_SqFt',
                'Year_Built', 'Floor_No', 'Total_Floors', 'Age_of_Property',
                'Nearby_Schools', 'Nearby_Hospitals']
print('\n✅ All numeric columns have correct data types.')

In [58]:
# Value range validation for key columns
print('Value Range Validation:')
print('-' * 50)
print(f'  BHK:              {df["BHK"].min()} - {df["BHK"].max()}')
print(f'  Size_in_SqFt:     {df["Size_in_SqFt"].min()} - {df["Size_in_SqFt"].max()}')
print(f'  Price_in_Lakhs:   {df["Price_in_Lakhs"].min()} - {df["Price_in_Lakhs"].max()}')
print(f'  Price_per_SqFt:   {df["Price_per_SqFt"].min():.4f} - {df["Price_per_SqFt"].max():.4f}')
print(f'  Year_Built:       {df["Year_Built"].min()} - {df["Year_Built"].max()}')
print(f'  Floor_No:         {df["Floor_No"].min()} - {df["Floor_No"].max()}')
print(f'  Total_Floors:     {df["Total_Floors"].min()} - {df["Total_Floors"].max()}')
print(f'  Age_of_Property:  {df["Age_of_Property"].min()} - {df["Age_of_Property"].max()}')
print(f'  Nearby_Schools:   {df["Nearby_Schools"].min()} - {df["Nearby_Schools"].max()}')
print(f'  Nearby_Hospitals: {df["Nearby_Hospitals"].min()} - {df["Nearby_Hospitals"].max()}')

Value Range Validation:
--------------------------------------------------
  BHK:              1 - 5
  Size_in_SqFt:     500 - 5000
  Price_in_Lakhs:   10.0 - 500.0
  Price_per_SqFt:   0.0000 - 0.9900
  Year_Built:       1990 - 2023
  Floor_No:         0 - 30
  Total_Floors:     1 - 30
  Age_of_Property:  2 - 35
  Nearby_Schools:   1 - 10
  Nearby_Hospitals: 1 - 10


In [ ]:
# Check for negative or zero values in columns that should be positive
print('Checking for invalid (zero or negative) values:')
for col in ['Size_in_SqFt', 'Price_in_Lakhs', 'Price_per_SqFt', 'Year_Built',
            'Floor_No', 'Total_Floors', 'Age_of_Property', 'Nearby_Schools', 'Nearby_Hospitals']:
    invalid = (df[col] <= 0).sum()
    status = '✅' if invalid == 0 else f'⚠️  {invalid:,} invalid values'
    print(f'  {col:25s} {status}')

## 9️⃣ Key Insights Summary

### Dataset Overview
- **250,000 property listings** across **20 states** and **42 cities**
- **23 columns** with **no missing values** and **no duplicate records**
- Data types are clean and consistent

### Target Variable (`Price_in_Lakhs`)
- Mean price: **~₹254.6 Lakhs**, Median: **~₹253.9 Lakhs**
- Distribution is **nearly symmetric** (skewness ≈ 0.008)
- Price range: **₹10L – ₹500L** with outliers detected via IQR method
- Log transformation does **not** improve symmetry (log skew = -1.22), so the original distribution is preferred

### Correlations
- **`Price_per_SqFt`** has the strongest correlation with price (r ≈ 0.56)
- **`Size_in_SqFt`** has a **very weak** linear correlation (r ≈ -0.003), suggesting price is not simply proportional to size — location and other factors dominate
- **`BHK`** also shows near-zero linear correlation, but price increases with BHK count (visible in box plots)

### Geospatial Patterns
- Prices are **relatively uniform** across cities (₹250.8L – ₹258.5L average)
- Bangalore is the most expensive city (₹258.5L avg), Cuttack the least (₹250.8L avg)
- State-level differences are similarly narrow

### Categorical Features
- **Property Type**: Independent House > Villa > Apartment
    - **Furnished Status**: Furnished > Semi-furnished > Unfurnished
    - **BHK**: Price increases with BHK count (1 BHK cheapest, 5 BHK most expensive)
    - **Public Transport**: High accessibility correlates with higher prices
    - **Parking & Security**: Properties with these amenities command higher prices

### Amenities
- 5 unique amenities: Playground, Gym, Garden, Pool, Clubhouse
- All amenities are present in most listings (high coverage)
- **Pool** and **Clubhouse** tend to have the largest positive price impact

### Data Quality
- ✅ No missing values
- ✅ No duplicate records
- ✅ All numeric columns have valid positive values
- ✅ Data types are correct and consistent

### Next Steps (Phase 3: Preprocessing)
- Handle price outliers (cap or remove extreme values)
- Multi-hot encode the `Amenities` column
- Target encode `Locality` and `City` (high cardinality)
- One-hot encode `Property_Type` and `Furnished_Status`
- Scale numerical features for model training